# T2-1. 타이타닉 Simple Baseline  

생존여부 예측모델 만들기  

학습용 데이터 (X_train, y_train) 을 이용하여 생존 예측 모형을 만든 후, 이를 평가용 데이터 (X_test) 에 적용하여 얻은 예측값을 다음과 같은 형식의 CSV 파일로 생성하시오.   
(가) 제공 데이터 목록
- y_train : 생존여부 (학습용)
- X_traink, X_test : 승객 정보 (학습용 및 평가용)

(나) 데이터 형식 및 내용  
- y_train (712명 데이터)  

유의사항  
- 성능이 우수한 예측모형을 구축하기 위해서는 적절한 데이터 전처리, 피처엔지니어링, 분류알고리즘, 하이퍼파라미터 튜닝, 모형 앙상블 등이 수반되어야한다.  
- 수험번호.csv 파일이 만들어지도록 코드를 제출한다.  
- 제출한 모델의 성능은 accuracy 로 평가함  

In [209]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split 

df = pd.read_csv(r"C:\Users\user\Desktop\GitHub\BigData_Analysis_Exam\Data\Titanic.csv")
display(df.head(5))

def exam_data_load(df, target, id_name ="", null_name = ""):
    # 데이터셋에 식별할 식별자가 없으면 (ID) 판다스가 기본적으로 부여하는 행 인덱스 (0,1,2,3..) 라도 강제로 부여
    if id_name == "":
        df=df.reset_index().rename(columns={"index","id"})
        id_name = 'id'
    else : 
        id_name = id_name 
    # ?,0,NaN 등의 비어있는 칸을 위해서 (통계 연산 불가능)
    # null_index 가 이상하면 정통 결측치 상태인 Nan 으로 치환해버린다.
    if null_name != "":
        df[df==null_name]=np.nan
    
    X_train,X_test = train_test_split(df ,test_size=0.2, random_state=42)

    # 무작위로 쪼개진 학습용 데이터 더이 X_train. 에서 채점에 꼭 필요한 'ID" 열과 '정답열만' 갖다가 쓰겟다.
    y_train = X_train[[id_name, target]]
    X_train = X_train.drop(columns=[target])

    y_test = X_test[[id_name, target]]
    X_test = X_test.drop(columns=[target])
    return X_train,X_test, y_train, y_test 

X_train,X_test, y_train, y_test = exam_data_load(df, target='Survived',id_name='PassengerId')

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [210]:
X_train.shape, y_train.shape, X_test.shape

((712, 11), (712, 2), (179, 11))

In [211]:
# EDA
X_train.head()

,PassengerId,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
331,332,1,"Partner, Mr. Austen",male,45.5,0,0,113043,28.5000,C124,S
733,734,2,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0000,NaN,S
382,383,3,"Tikkanen, Mr. Juho",male,32.0,0,0,STON/O 2. 3101293,7.9250,NaN,S
704,705,3,"Hansen, Mr. Henrik Juul",male,26.0,1,0,350025,7.8542,NaN,S
813,814,3,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4,2,347082,31.2750,NaN,S


In [212]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 331 to 102
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Pclass       712 non-null    int64  
 2   Name         712 non-null    str    
 3   Gender       712 non-null    str    
 4   Age          572 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Ticket       712 non-null    str    
 8   Fare         712 non-null    float64
 9   Cabin        159 non-null    str    
 10  Embarked     710 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 66.8 KB


In [213]:
y_train.head()

,PassengerId,Survived
331,332,0
733,734,0
382,383,0
704,705,0
813,814,0


In [214]:
y_train['Survived'].value_counts() # 생존 비율

Survived
0    444
1    268
Name: count, dtype: int64

In [215]:
# 데이터 전처리
y = y_train['Survived']

# sex만 원핫 인코딩 웹
features = ["Pclass","Gender","SibSp","Parch"]
# get_dummies : 데이터 프레임을 통째로 집어넣으면, 
# 내부에 잇는 열 중 object 타입 (문자열) 이나 category 타입의 열만 자동으로 골라내 원-핫 인코딩 수행
# 숫자형 열은 그대로 둔다. 
display(X_train.head(2))
X= pd.get_dummies(X_train[features])
display(X.head())
test = pd.get_dummies(X_test[features])

,PassengerId,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
331,332,1,"Partner, Mr. Austen",male,45.5,0,0,113043,28.5,C124,S
733,734,2,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0,NaN,S


,Pclass,SibSp,Parch,Gender_female,Gender_male
331,1,0,0,False,True
733,2,0,0,False,True
382,3,0,0,False,True
704,3,1,0,False,True
813,3,4,2,True,False


In [216]:
X.shape, test.shape

((712, 5), (179, 5))

In [217]:
# 모델 및 평가
from sklearn.ensemble import RandomForestClassifier 

model1 = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42)
model1.fit(X,y)
predictions=model1.predict(test)

model1.score(X,y)

0.8230337078651685

In [218]:
# 모델 및 평가
from sklearn.ensemble import RandomForestClassifier 

model2 = RandomForestClassifier(n_estimators=1000, max_depth=7, random_state=42)
model2.fit(X,y)
predictions=model2.predict(test)

model2.score(X,y)

0.824438202247191

In [219]:
# 모델 및 평가
from sklearn.ensemble import RandomForestClassifier 

model3 = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
model3.fit(X,y)
predictions=model3.predict(test)

model3.score(X,y)

0.824438202247191

In [220]:
# 모델 및 평가
from sklearn.ensemble import RandomForestClassifier 

model4 = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=1000)
model4.fit(X,y)
predictions=model4.predict(test)

model4.score(X,y)

0.824438202247191

In [221]:
output = pd.DataFrame({"PassengerId":X_test.PassengerId, "Survived":predictions})
output.head()

,PassengerId,Survived
709,710,0
439,440,0
840,841,0
720,721,1
39,40,0


In [222]:
# 수험번호.csv 로 출력

"""
output.to_csv("123.csv",index=False)

"""

'\noutput.to_csv("123.csv",index=False)\n\n'

In [223]:
# 결과 채점
print(model1.score(test,y_test["Survived"]))
print(model2.score(test,y_test["Survived"]))
print(model3.score(test,y_test["Survived"]))
print(model4.score(test,y_test["Survived"]))

0.7821229050279329
0.7821229050279329
0.7877094972067039
0.7821229050279329


In [224]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier 

# 1. 원-핫 인코딩이 완료된 X 데이터를 학습용(train)과 검증용(val)으로 쪼갭니다.
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 모델 선언 및 학습 (학습용 데이터만 사용!)
model = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42)
model.fit(X_tr, y_tr)

# 3. 🔥 성능 평가는 반드시 검증용 데이터(X_val, y_val)로 확인!
print("훈련 데이터 점수:", model.score(X_tr, y_tr))
print("검증 데이터 점수:", model.score(X_val, y_val))

훈련 데이터 점수: 0.8312829525483304
검증 데이터 점수: 0.7902097902097902


# T2-2 Pima Indians Diabetes (피아 인디언 당뇨병)  

당뇨병 여부 판단  
예측 컬럼 : Outcome (0 정상, 1 당뇨) 당뇨병일 확률 예측  
평가지표 : roc-auc  
제출파일명 : result.csv(1개 컬럼, 컬럼명 pred)  

In [225]:
import pandas as pd 
import os 
from sklearn.model_selection import train_test_split  

train = pd.read_csv(r"C:\Users\user\Desktop\GitHub\BigData_Analysis_Exam\Data\diabetes_train.csv")
test = pd.read_csv(r"C:\Users\user\Desktop\GitHub\BigData_Analysis_Exam\Data\diabetes_test.csv")
display(train.head(2))
display(test.head(2))


,id,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,377,1,87,60,37,75,37.2,0.509,22,0
1,370,3,173,82,48,465,38.4,2.137,25,1


,id,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,258,1,193,50,16,375,25.9,0.655,24
1,220,0,177,60,29,478,34.6,1.072,21


In [226]:
train['Outcome'].value_counts()

Outcome
0    381
1    195
Name: count, dtype: int64

In [227]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 576 entries, 0 to 575
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        576 non-null    int64  
 1   Pregnancies               576 non-null    int64  
 2   Glucose                   576 non-null    int64  
 3   BloodPressure             576 non-null    int64  
 4   SkinThickness             576 non-null    int64  
 5   Insulin                   576 non-null    int64  
 6   BMI                       576 non-null    float64
 7   DiabetesPedigreeFunction  576 non-null    float64
 8   Age                       576 non-null    int64  
 9   Outcome                   576 non-null    int64  
dtypes: float64(2), int64(8)
memory usage: 45.1 KB


In [228]:
train.isnull().sum()

id                          0
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [229]:
# 데이터 전처리
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age"]
train[cols] = scaler.fit_transform(train[cols])
test[cols] = scaler.fit_transform(test[cols])

'''
StandardScaler : 서로 다른 척도 (단위) 를 가진 데이터들의 체급을 똑같이 맞춰주는 도구
데이터의 평균을 0, 표준편차를 1이 되도록 변환하여 정규분포 모양으로 표준화하는 역할을 한다.  
ex) 인슐린 숫자는 몇백단위로 요동치는데 당뇨 내력 가중치는 1 안팍에서 놀아 변수의 가중치가 조정되는 것을 고치기 위함.
'''

'\nStandardScaler : 서로 다른 척도 (단위) 를 가진 데이터들의 체급을 똑같이 맞춰주는 도구\n데이터의 평균을 0, 표준편차를 1이 되도록 변환하여 정규분포 모양으로 표준화하는 역할을 한다.  \nex) 인슐린 숫자는 몇백단위로 요동치는데 당뇨 내력 가중치는 1 안팍에서 놀아 변수의 가중치가 조정되는 것을 고치기 위함.\n'

In [230]:
# id 제외
display(train.head(2))
train = train.drop('id',axis=1)
test = test.drop('id',axis=1)
display(train.head(2))

,id,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,377,-0.846722,-1.037009,-0.419767,1.000802,-0.061847,0.687444,0.086445,-0.949139,0
1,370,-0.255914,1.651637,0.676768,1.691778,3.319349,0.841447,4.828849,-0.698357,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,-0.846722,-1.037009,-0.419767,1.000802,-0.061847,0.687444,0.086445,-0.949139,0
1,-0.255914,1.651637,0.676768,1.691778,3.319349,0.841447,4.828849,-0.698357,1


In [231]:
target = train.pop("Outcome")
display(target)

0      0
1      1
2      1
3      1
4      1
      ..
571    0
572    1
573    0
574    0
575    1
Name: Outcome, Length: 576, dtype: int64

In [232]:
# 학습 및 예측
from sklearn.ensemble import RandomForestClassifier 

model = RandomForestClassifier(random_state=42)
model.fit(train,target)
predictions = model.predict_proba(test)
display(predictions)
# 당뇨 확률값 선택
pred = predictions[:,1]

array([[0.39, 0.61],
       [0.28, 0.72],
       [1.  , 0.  ],
       [0.15, 0.85],
       [0.68, 0.32],
       [0.78, 0.22],
       [0.3 , 0.7 ],
       [0.98, 0.02],
       [0.71, 0.29],
       [0.71, 0.29],
       [0.57, 0.43],
       [0.91, 0.09],
       [0.95, 0.05],
       [0.4 , 0.6 ],
       [0.97, 0.03],
       [0.54, 0.46],
       [0.76, 0.24],
       [0.35, 0.65],
       [0.46, 0.54],
       [0.44, 0.56],
       [0.78, 0.22],
       [0.19, 0.81],
       [0.97, 0.03],
       [0.23, 0.77],
       [0.6 , 0.4 ],
       [0.77, 0.23],
       [0.28, 0.72],
       [0.9 , 0.1 ],
       [0.71, 0.29],
       [0.25, 0.75],
       [0.71, 0.29],
       [0.79, 0.21],
       [0.25, 0.75],
       [0.98, 0.02],
       [0.89, 0.11],
       [0.77, 0.23],
       [0.97, 0.03],
       [0.98, 0.02],
       [0.93, 0.07],
       [0.58, 0.42],
       [0.64, 0.36],
       [0.89, 0.11],
       [0.54, 0.46],
       [0.69, 0.31],
       [0.88, 0.12],
       [0.8 , 0.2 ],
       [0.78, 0.22],
       [0.95,

In [233]:
output = pd.DataFrame({'pred':pred})
output.head()

,pred
0,0.61
1,0.72
2,0.00
3,0.85
4,0.32


# T2-3. Adult Census Income Tutorial  
성인 인구조사 소득 예측 
- age(나이), workclass (고용 형태), fnlwgt (사람의 대표성을 나타내는 가중치, final weight), education (교육 수준), education.num (교육 수준 수치), marital.status (결혼 상태), occupation (업종), relationship (가족 관계), race (인종), sex (성별), capital.gain (양도 소득), capital.loss (양도 손실), hours.per.week (주당 근무 시간), native.country (국적), income (수익, 예측해야하는 값)

In [234]:
import pandas as pd 
from sklearn.model_selection import train_test_split 

df = pd.read_csv(r"C:\Users\user\Desktop\GitHub\BigData_Analysis_Exam\Data\adult.csv")
display(df.head(2))

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K


In [235]:
# 시험환경 세팅 
def exam_data_load(df, target, id_name="", null_name=""):
    if id_name == "":
        df = df.reset_index().rename(columns={"index": "id"})
        id_name = 'id'
    else:
        id_name = id_name
    
    if null_name != "":
        df[df == null_name] = np.nan
    
    X_train, X_test = train_test_split(df, test_size=0.2, random_state=2021)
    
    y_train = X_train[[id_name, target]]
    X_train = X_train.drop(columns=[target])

    
    y_test = X_test[[id_name, target]]
    X_test = X_test.drop(columns=[target])
    return X_train, X_test, y_train, y_test 

In [236]:
X_train, X_test, y_train, y_test = exam_data_load(df,target='income',null_name="?")
display(X_train.head(2))
display(X_test.head(2))
display(y_train.head(2))
display(y_test.head(2))

,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
21851,21851,36,Private,241998,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,50,United-States
7632,7632,53,Private,103950,Masters,14,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,40,United-States


,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
20901,20901,58,Private,114495,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States
14170,14170,46,Private,247043,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States


,id,income
21851,21851,>50K
7632,7632,<=50K


,id,income
20901,20901,<=50K
14170,14170,>50K


In [237]:
# EDA
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
display(X_train.head(2))

(26048, 15) (6513, 15) (26048, 2) (6513, 2)


,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
21851,21851,36,Private,241998,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,50,United-States
7632,7632,53,Private,103950,Masters,14,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,40,United-States


In [238]:
print(y_train['income'].value_counts())

income
<=50K    19756
>50K      6292
Name: count, dtype: int64


In [239]:
print(X_train.isnull().sum())

id                   0
age                  0
workclass         1456
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1463
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     461
dtype: int64


In [240]:
print(X_test.isnull().sum())

id                  0
age                 0
workclass         380
fnlwgt              0
education           0
education.num       0
marital.status      0
occupation        380
relationship        0
race                0
sex                 0
capital.gain        0
capital.loss        0
hours.per.week      0
native.country    122
dtype: int64


In [241]:
numeric_features = ["age","fnlwgt","education.num","capital.gain","capital.loss","hours.per.week"]
str_features = ["workclass","education","marital.status","occupation","relationship","race","sex","native.country"]

print(X_train.isnull().sum())
# 결측치 처리 (최빈값과 차이가 크면 최빈값으로 값이 비슷하면 별도의 값으로)
def data_fillna(df) : 
    df['workclass'] = df['workclass'].fillna(df['workclass'].mode()[0])
    df['occupation'] = df['occupation'].fillna("null")
    df["native.country"] = df["native.country"].fillna(df["native.country"].mode()[0])
    return df

X_train = data_fillna(X_train)
X_test = data_fillna(X_test)

id                   0
age                  0
workclass         1456
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1463
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     461
dtype: int64


In [242]:
print(X_train.isnull().sum())

id                0
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
dtype: int64


In [243]:
# 피쳐 엔지니어링
from sklearn.preprocessing import LabelEncoder 

all_df = pd.concat([X_train.assign(ind="train"),X_test.assign(ind="test")])
le = LabelEncoder()
all_df[str_features] = all_df[str_features].apply(le.fit_transform)

X_train = all_df[all_df['ind']=="train"]
X_train = X_train.drop('ind',axis=1)
display(X_train)

X_test= all_df[all_df['ind']=="test"]
X_test = X_test.drop('ind',axis=1)
display(X_test)

,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
21851,21851,36,3,241998,9,13,2,2,0,4,1,0,0,50,38
7632,7632,53,3,103950,12,14,0,9,1,4,0,0,0,40,38
27878,27878,19,3,203061,15,10,4,12,1,4,0,0,0,25,38
14121,14121,20,3,102607,11,9,4,5,3,4,1,0,0,30,38
32345,32345,54,6,138852,11,9,2,9,0,4,1,0,0,40,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2669,2669,45,3,187370,12,14,0,3,4,4,1,7430,0,70,38
17536,17536,36,3,174308,1,7,0,13,1,4,1,0,0,40,38
6201,6201,47,3,275361,7,12,6,7,3,4,0,0,0,35,38
27989,27989,50,5,196504,10,16,2,9,0,4,1,0,0,23,38


,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
20901,20901,58,3,114495,11,9,2,3,0,4,1,0,0,40,38
14170,14170,46,3,247043,11,9,2,13,0,4,1,0,0,40,38
1776,1776,67,1,103315,12,14,4,3,2,4,0,15831,0,72,38
30428,30428,18,3,165532,15,10,4,11,3,4,1,0,0,15,38
8602,8602,26,6,58039,15,10,2,7,0,4,1,0,0,40,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31222,31222,22,3,199426,15,10,4,14,1,4,0,0,0,40,38
10861,10861,41,3,155106,11,9,2,5,0,4,1,0,0,40,38
8929,8929,32,3,153078,9,13,4,7,1,1,1,0,0,40,34
2066,2066,48,6,171926,14,15,2,9,0,4,1,15024,0,50,38


In [244]:
# 스케일링
from sklearn.preprocessing import MinMaxScaler 

scaler = MinMaxScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.fit_transform(X_test[numeric_features])
display(X_train.head(2))

,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
21851,21851,0.260274,3,0.156011,9,0.800000,2,2,0,4,1,0.0,0.0,0.500000,38
7632,7632,0.493151,3,0.062255,12,0.866667,0,9,1,4,0,0.0,0.0,0.397959,38


In [245]:
y = (y_train['income'] != "<=50K").astype(int)
y[:5]

21851    1
7632     0
27878    0
14121    0
32345    0
Name: income, dtype: int64

In [247]:
# 검증용 데이터 분리
from sklearn.model_selection import train_test_split 
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.15, random_state=42)
display(X_tr.head(2))

,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
12462,12462,0.136986,3,0.017192,15,0.600000,4,7,1,4,1,0.0,0.0,0.234694,38
15837,15837,0.287671,3,0.112052,11,0.533333,2,14,5,4,0,0.0,0.0,0.346939,38


In [248]:
X_tr = X_tr.drop('id',axis=1)
X_val = X_val.drop('id',axis=1)

In [250]:
# 모델 & 평가
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(random_state=42)
model.fit(X_tr,y_tr)
pred= model.predict(X_val)
print("accuracy score : ",(accuracy_score(y_val, pred)))

accuracy score :  0.8011770726714432


In [251]:
# 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state = 42)
model.fit(X_tr,y_tr)
pred = model.predict(X_val)
print("accuracy score : ",(accuracy_score(y_val, pred)))

accuracy score :  0.848259979529171


In [252]:
# test 데이터 예측
X_test_id = X_test.pop('id')
pred = model.predict(X_test)

In [253]:
output = pd.DataFrame({'id':X_test_id, 'income' : pred})
output.head(5)

,id,income
20901,20901,1
14170,14170,0
1776,1776,1
30428,30428,0
8602,8602,0


In [254]:
# 채점
y_test = (y_test['income'] != '<=50K').astype(int)
from sklearn.metrics import accuracy_score 
print("accuracy score : ",(accuracy_score(y_test,pred)))

accuracy score :  0.8453861507753724
